# Lesson 02 - 探索 Microsoft Agent Framework

**Microsoft Agent Framework (MAF)** 是一個用於構建 AI 代理的統一框架。它提供了一個乾淨且可組合的架構，包含四個核心構建模塊：

- **Client** – 連接到 AI 模型端點並處理通信
- **Agent** – 以指令和工具定義包裝客戶端
- **Tools** – 使用模型可以調用的自訂函數擴展代理功能
- **Session** – 維護多回合交互的對話歷史

在本課中，我們將使用這些概念構建一個**旅遊預訂代理**，用來檢查目的地的可用性。


## 設定


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## 了解 Agent 框架架構

Microsoft Agent 框架採用分層架構：

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **Client** – 一個 `AzureAIProjectAgentProvider` 連接到 Azure OpenAI 部署。它負責驗證、請求格式化和回應解析。
2. **Agent** – 透過 `provider.create_agent()` 從 client 創建的 agent，將模型存取、指令（系統提示）和工具結合在一起。
3. **Tools** – 使用 `@tool` 裝飾的 Python 函式，agent 可以調用它們來執行動作或擷取資料。
4. **Session** – 一個 `AgentSession` 物件（透過 `agent.create_session()` 創建），用來儲存對話歷史，實現多輪對話，讓 agent 記住先前的上下文。

讓我們一步步建立每一層。


In [ ]:
# Create the client – this is the connection to the AI model
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## 使用 @tool 裝飾器新增工具

工具讓代理人能進行產生文字以外的操作。`@tool` 裝飾器將一般的 Python 函式轉換成代理人可以呼叫的功能。

重點：
- 使用 `Annotated[type, "description"]` 讓模型理解每個參數。
- 文件字串會成為模型看到的工具說明。
- `approval_mode="never_require"` 意味著該工具會自動執行，無需用戶確認。


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## 建立具工具的代理人

現在我們結合客戶端、指令和工具成為代理人。`instructions` 作為系統提示 — 定義代理人的人格和行為。


In [ ]:
agent = await provider.create_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## 使用會話進行多輪對話

一個 `AgentSession`（透過 `agent.create_session()` 建立）會追蹤對話中的所有訊息。透過在每次呼叫 `agent.run()` 時傳入相同的會話，代理可以存取完整的對話記錄，並參考較早的訊息。

我們傳入 `tools=[check_destination_availability]`，使代理在每一輪對話中可以呼叫我們的可用性檢查工具。


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## Summary

在本課程中，您探討了 Microsoft Agent Framework 的四大支柱：

| Concept | What You Learned |
|---------|------------------|
| **Client** | `AzureAIProjectAgentProvider` 透過基於憑證的身份驗證連接到 Azure OpenAI |
| **Agent** | `provider.create_agent()` 將模型連接、指令和名稱打包在一起 |
| **Tools** | `@tool` 裝飾器開放 Python 函數供代理調用 |
| **Session** | `agent.create_session()` 在多輪對話中維持會話歷史 |

這些構建塊結合起來，創建能夠進行自然對話、調用外部函數並維持上下文的代理 — 為後續課程中更高階的代理模式奠定基礎。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：  
本文件係使用 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 進行翻譯。雖然我們致力於翻譯準確性，但請注意自動翻譯可能包含錯誤或不準確之處。原始文件的母語版本應視為權威來源。對於重要資訊，建議採用專業人工翻譯。對於因使用本翻譯所產生之任何誤解或誤譯，我們不承擔任何責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
